# Tool-use trace generator (laptop **or** Colab)

Agentic distillation: the teacher (**DeepSeek V4 Flash**) solves problems with **real tools** (`python` sandbox + Brave `search`) via native function-calling. Tools execute for real; only traces whose final answer **verifies against a known gold** (and that used ≥1 tool, fit the 2048 budget, and had no failed search) are kept. Output: the `tool_use` SFT source.

**Two skills, two pools:**
- **`python`** (cells 6–7): compute-offload on math problems (orca/numina). High yield, no Brave spend.
- **`search` = fact lookup** (cells 8–9): build a **verifiable fact DB from Wikidata** (cell 8), then have the teacher look each fact up on Brave and verify against the Wikidata gold (cell 9). Search is *forced* first so the teacher actually looks up rather than answering from memory. This is the achievable, 2B-friendly form of search (copy a value, not summarize a method).

Run calibration (`--limit`) before any full run.

> ⚠️ On Colab this clones the repo from GitHub — **push the latest `sft/` first**. Needs `tokenizer_out/tokenizer.json` under `SYNAPSE_DIR` on Drive (fingerprint `7a570a7ba9fc7985`).

In [ ]:
# 1. Detect environment + set SYNAPSE_DIR
import os
try:
    from google.colab import drive
    COLAB = True
    drive.mount('/content/drive', force_remount=False)
    SYNAPSE_DIR = '/content/drive/MyDrive/synapse'
except ImportError:
    COLAB = False
    SYNAPSE_DIR = os.environ.get('SYNAPSE_DIR') or os.path.abspath('./synapse')
os.environ['SYNAPSE_DIR'] = SYNAPSE_DIR
print('COLAB =', COLAB, '| SYNAPSE_DIR =', SYNAPSE_DIR)

In [ ]:
# 2. Deps (tokenizers is required by tools_runtime for token-exact budgeting)
!pip install -q openai datasets sympy python-dotenv tqdm tokenizers

In [ ]:
# 3. Locate the repo (clone on Colab; find it locally on laptop)
import os, subprocess
if COLAB:
    REPO_DIR = '/content/synapse_repo'
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'], check=True)
    else:
        subprocess.run(['git','clone','--depth=1','https://github.com/ajencinas/synapse.git',REPO_DIR], check=True)
else:
    d = os.path.abspath('.')
    while d != os.path.dirname(d) and not os.path.isfile(os.path.join(d,'sft','generate_tool_use.py')):
        d = os.path.dirname(d)
    REPO_DIR = d
assert os.path.isfile(os.path.join(REPO_DIR,'sft','generate_tool_use.py')), \
    'generate_tool_use.py not found — push it to GitHub (Colab) or run from inside the repo (laptop)'
print('REPO_DIR =', REPO_DIR)

In [ ]:
# 4. API keys.
#    Teacher: OPENROUTER_API_KEY *or* DEEPSEEK_API_KEY (auto-detected; DeepSeek preferred).
#    Search:  BRAVE_API_KEY (only needed for --mode search).
#    Colab: add them in Secrets (🔑). Laptop: read from repo .env automatically.
import os
if COLAB:
    try:
        from google.colab import userdata
        for name in ('OPENROUTER_API_KEY','DEEPSEEK_API_KEY','BRAVE_API_KEY'):
            if not os.environ.get(name):
                v = userdata.get(name)
                if v: os.environ[name] = v
    except Exception:
        pass
teacher = [n for n in ('DEEPSEEK_API_KEY','OPENROUTER_API_KEY') if os.environ.get(n)]
print('teacher keys:', teacher or 'none yet — will fall back to repo .env')
print('BRAVE_API_KEY:', 'set' if os.environ.get('BRAVE_API_KEY') else 'NOT set (search mode will fail)')

In [ ]:
# 5. Sanity check: tokenizer present + dedicated tool tokens are 7/8 (fail loud now).
#    Also smoke-tests the python sandbox.
import sys; sys.path.insert(0, os.path.join(REPO_DIR,'sft'))
import importlib, tools_runtime as tr; importlib.reload(tr)
tr.verify_tool_tokens()
print('tool tokens OK: <|tool_call|>=7, <|tool_result|>=8')
print('python sandbox:', tr.run_python('print(2847*391)'))

## A) `python` tool — compute offload (cells 6–7)

In [ ]:
# 6. CALIBRATION — python mode, 1.5k problems. Cheap, deterministic, no Brave spend.
#    Prints yield% + projected full cost; inspect before a full run.
cmd = f'cd {REPO_DIR} && python sft/generate_tool_use.py --mode python --limit 1500 --workers 48'
print(cmd)
!{cmd}

In [ ]:
# 7. FULL RUN — python mode (uncapped; add --budget-usd N to hard-cap). Resumable:
#    re-run to continue (no dupes, no re-spend). Saved per-trace on Drive.
cmd = f'cd {REPO_DIR} && python sft/generate_tool_use.py --mode python --workers 48'
print(cmd)
!{cmd}

## B) `search` tool — fact lookup (cells 8–9)
Build a verifiable fact DB from Wikidata (gold = the Wikidata value), then the teacher looks each fact up on Brave and we keep it only if the answer matches the gold.

In [ ]:
# 8. Build the fact-lookup DB from Wikidata — ALL 13 families (diverse domains),
#    -n 8000 target. ~13 queries, each retrying ~60s through Wikidata's 1-req/min
#    throttle, so budget ~10-20 min. Overwrites cleanly (deduped, no dupes).
FACTS = f'{SYNAPSE_DIR}/datasets_sft/tool_use/facts_problems.jsonl'
cmd = (f'cd {REPO_DIR} && python sft/generate_tool_problems.py --kind facts --source wikidata '
       f'-n 8000 --per-family-fetch 6000 --retry-wait 60 --retries 8 --out {FACTS}')
print(cmd)
!{cmd}
print('\nsample:'); print(open(FACTS).readline().strip())

In [ ]:
# 9. RUN — search mode over the fact DB. Brave is HARD-LIMITED to 1 req/s
#    (--search-rate 1.0); responses are scanned for error envelopes and rejected.
#    Forces a search first, verifies vs the Wikidata gold. Needs BRAVE_API_KEY.
#    ~1 search/problem at 1/s -> budget ~1 hr per ~3-4k problems.
cmd = (f'cd {REPO_DIR} && python sft/generate_tool_use.py --mode search '
       f'--problems {FACTS} --workers 16 --search-rate 1.0')
print(cmd)
!{cmd}

In [ ]:
# 10. Inspect output (works for both python and search traces)
import json, os
base = os.path.join(SYNAPSE_DIR, 'datasets_sft', 'tool_use')
raw = os.path.join(base, 'tool_use_raw.jsonl')
n = sum(1 for _ in open(raw)) if os.path.exists(raw) else 0
print('kept traces:', n)
if n:
    ex = json.loads(open(raw).readline())
    print('mode:', ex.get('mode'), '| turns:', len(ex['messages']))
    for m in ex['messages']:
        tag = m['role'] + ('+tool_call' if 'tool_call' in m else '')
        body = m.get('tool_call', m.get('content',''))
        print(f"  [{tag}] {str(body)[:160]}")
mp = os.path.join(base, 'meta_raw.json')
if os.path.exists(mp):
    print('\nmeta:', json.load(open(mp)))

## After generating → fold into SFT
```bash
python sft/tokenize_sft_data.py --datasets tool_use --force   # needs the encode_example change (PLAN §3)
python sft/consolidate_sft_data.py
```
Then add `"tool_use": ~0.12` to `SFT_DATA_MIX` in `sft.py` **after** tokenizing (the trainer hard-fails on a mix source with no `train.jsonl`).

**Notes**
- Resumable: `progress.log` tracks processed ids — re-run any cell if Colab drops (no dupes, no re-spend). python and search ids don't collide, so both append to the same `tool_use_raw.jsonl`.
- `search` forces a search call first (teacher knows many facts → would skip the tool otherwise); kept only if the answer matches the Wikidata gold and no `[search unavailable]`.
- Offline alternative to cell 8: `generate_tool_problems.py --source seed --seed my_facts.jsonl` with your own `{question, gold}` lines.
- GSM8K excluded + decontaminated on the python pool.